In [0]:
# COMMAND ----------

import json
import io
import csv
import zipfile
import datetime as dt
from typing import Dict, Any, Optional

import requests

# COMMAND ----------

# ---------------- Config ----------------
CATALOG = "canada_business"
SCHEMA  = "bronze"
BRONZE_SUBDIR = "statcan_raw"

TABLES: Dict[str, str] = {
    "20100056": "Monthly retail trade sales by province and territory",
    "20100076": "Retail trade inventories: inventory to sales ratio",
    "12100121": "International merchandise trade by commodity (NAPCS)",
}

STATCAN_BASE = "https://www150.statcan.gc.ca/t1/wds/rest"

# COMMAND ----------

# ---------------- Mode ----------------
dbutils.widgets.dropdown("MODE", "incremental", ["incremental", "backfill"])
MODE = dbutils.widgets.get("MODE").strip().lower()
print(f"MODE: {MODE}")

# COMMAND ----------

# ---------------- Helpers ----------------
def run_ts_utc() -> str:
    return dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def resolve_schema_root_location(catalog: str, schema: str) -> str:
    df = spark.sql(f"DESCRIBE SCHEMA EXTENDED {catalog}.{schema}").toPandas()
    rows = df.loc[df["database_description_item"] == "RootLocation", "database_description_value"].values
    if len(rows) == 0 or not rows[0]:
        raise RuntimeError(f"RootLocation not found for {catalog}.{schema}")
    return str(rows[0])

def join_uri(base_uri: str, child: str) -> str:
    return base_uri.rstrip("/") + "/" + child.strip("/")


def get_csv_download_url(pid: str) -> str:
    """Call StatCan API to get temporary ZIP download URL."""
    url = f"{STATCAN_BASE}/getFullTableDownloadCSV/{pid}/en"
    headers = {"User-Agent": "databricks-lakehouse-demo/1.0"}
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    payload = r.json()
    if payload.get("status") != "SUCCESS":
        raise RuntimeError(f"StatCan API error for PID {pid}: {payload}")
    return payload["object"]


def download_and_extract_csv(zip_url: str) -> Dict[str, str]:
    """Download ZIP, extract CSVs, return {filename: content_string}."""
    headers = {"User-Agent": "databricks-lakehouse-demo/1.0"}
    r = requests.get(zip_url, headers=headers, timeout=300)
    r.raise_for_status()
    files = {}
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        for name in zf.namelist():
            if name.endswith(".csv"):
                files[name] = zf.read(name).decode("utf-8-sig")
    return files


def filter_new_months(csv_content: str, max_ref_date: Optional[str]):
    """
    Filter CSV to rows with REF_DATE > max_ref_date.
    Returns (filtered_csv_string, new_row_count, new_max_ref_date).
    Returns (None, 0, None) if no new data.
    """
    reader = csv.reader(io.StringIO(csv_content))
    header = next(reader)

    try:
        ref_date_idx = header.index("REF_DATE")
    except ValueError:
        # MetaData file — no REF_DATE column
        return None, 0, None

    if max_ref_date is None:
        # Backfill: keep everything, find max date
        all_rows = list(reader)
        new_max = max(
            (r[ref_date_idx] for r in all_rows if len(r) > ref_date_idx and r[ref_date_idx]),
            default=None
        )
        return csv_content, len(all_rows), new_max

    # Incremental: filter to new months only
    new_rows = []
    new_max = max_ref_date
    for row in reader:
        if len(row) > ref_date_idx:
            ref_date = row[ref_date_idx]
            if ref_date > max_ref_date:
                new_rows.append(row)
                if ref_date > new_max:
                    new_max = ref_date

    if not new_rows:
        return None, 0, None

    output = io.StringIO()
    writer = csv.writer(output)
    writer.writerow(header)
    writer.writerows(new_rows)
    return output.getvalue(), len(new_rows), new_max

# COMMAND ----------

# ---------------- Read watermarks ----------------
watermarks = {}
if MODE == "incremental":
    wm_df = spark.sql("""
        SELECT series_key AS pid, last_obs_date
        FROM canada_business.bronze.ingest_watermarks
        WHERE source = 'statcan'
    """).toPandas()
    for _, row in wm_df.iterrows():
        if row["last_obs_date"] is not None:
            watermarks[row["pid"]] = row["last_obs_date"].strftime("%Y-%m")
    print(f"Loaded watermarks for {len(watermarks)} PIDs:")
    for k, v in watermarks.items():
        print(f"  PID {k}: last REF_DATE = {v}")

# COMMAND ----------

# ---------------- Resolve paths ----------------
root_location = resolve_schema_root_location(CATALOG, SCHEMA)
bronze_base = join_uri(root_location, BRONZE_SUBDIR)
data_base = bronze_base.rstrip("/") + "/data"
manifest_base = bronze_base.rstrip("/") + "/manifests"

dbutils.fs.mkdirs(data_base)
dbutils.fs.mkdirs(manifest_base)

ts = run_ts_utc()
print(f"Run ts: {ts} | MODE: {MODE}")

# COMMAND ----------

# ---------------- Ingest ----------------
manifest = {
    "run_ts": ts,
    "catalog": CATALOG,
    "schema": SCHEMA,
    "bronze_base": bronze_base,
    "source": "Statistics Canada Full Table Download (CSV)",
    "mode": MODE,
    "tables": []
}

for pid, description in TABLES.items():
    print(f"\n⏳ PID {pid}: {description}")

    # Step 1: Get download URL
    zip_url = get_csv_download_url(pid)
    print(f"  ZIP URL: {zip_url}")

    # Step 2: Download ZIP and extract
    csv_files = download_and_extract_csv(zip_url)

    table_manifest = {
        "pid": pid,
        "description": description,
        "zip_url": zip_url,
        "mode": MODE if MODE == "backfill" else ("incremental" if pid in watermarks else "backfill"),
        "files": []
    }

    out_dir = join_uri(data_base, pid)
    dbutils.fs.mkdirs(out_dir)

    # In backfill mode, delete old data files first
    if MODE == "backfill":
        try:
            for f in dbutils.fs.ls(out_dir):
                if f.name.endswith(".csv"):
                    dbutils.fs.rm(f.path)
            print(f"  🗑️ Cleaned old files for PID {pid}")
        except Exception:
            pass

    max_ref_date_written = None

    for filename, content in csv_files.items():
        # Skip MetaData files
        if "MetaData" in filename:
            continue

        # Step 3: Filter based on mode
        if MODE == "backfill":
            max_ref = None  # write everything
        else:
            max_ref = watermarks.get(pid, None)

        filtered_content, new_rows, new_max = filter_new_months(content, max_ref)

        if filtered_content is None or new_rows == 0:
            print(f"  ⏭️ No new months in {filename}")
            continue

        # Step 4: Write
        out_file = join_uri(out_dir, f"{filename.replace('.csv', '')}_{ts}.csv")
        dbutils.fs.put(out_file, filtered_content, overwrite=False)

        label = "backfill" if max_ref is None else "incremental"
        print(f"  ✅ [{label}] {filename} -> {new_rows} rows -> {out_file}")

        table_manifest["files"].append({
            "original_filename": filename,
            "raw_file": out_file,
            "new_rows": new_rows,
            "bytes": len(filtered_content)
        })

        if new_max and (max_ref_date_written is None or new_max > max_ref_date_written):
            max_ref_date_written = new_max

    # Update watermark
    if max_ref_date_written:
        wm_date = f"{max_ref_date_written}-01"
        total_rows = sum(f.get("new_rows", 0) for f in table_manifest["files"])

        spark.sql(f"""
            MERGE INTO canada_business.bronze.ingest_watermarks AS t
            USING (SELECT
                'statcan' AS source, '{pid}' AS series_key,
                DATE('{wm_date}') AS last_obs_date, '{ts}' AS last_run_ts,
                {total_rows} AS rows_written, current_timestamp() AS updated_at
            ) AS s
            ON t.source = s.source AND t.series_key = s.series_key
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

    manifest["tables"].append(table_manifest)

# Write manifest
manifest_path = join_uri(manifest_base, f"run_manifest_{ts}.json")
dbutils.fs.put(manifest_path, json.dumps(manifest, indent=2), overwrite=False)

print(f"\n✅ StatCan ingest complete ({MODE})")
print("Manifest:", manifest_path)
for t in manifest["tables"]:
    total = sum(f.get("new_rows", 0) for f in t["files"])
    print(f"  PID {t['pid']}: {total} rows ({t['mode']})")




























